# Faz 3 — Feature Engineering


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('../data/processed/merged.csv', parse_dates=['Date'])
print(df.shape)


In [26]:
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['WeekOfYear'] = df['Date'].dt.isocalendar().week
df['Quarter'] = df['Date'].dt.quarter

for lag in [7, 14, 28]:
    df[f'Sales_lag_{lag}'] = df.groupby('Store')['Sales'].shift(lag)

df[['Store', 'Date', 'Open', 'Sales', 'Sales_lag_7', 'Sales_lag_14', 'Sales_lag_28']].head(15)

,Store,Date,Open,Sales,Sales_lag_7,Sales_lag_14,Sales_lag_28
0,1,2013-01-01,0,0,NaN,NaN,NaN
1,1,2013-01-02,1,5530,NaN,NaN,NaN
2,1,2013-01-03,1,4327,NaN,NaN,NaN
3,1,2013-01-04,1,4486,NaN,NaN,NaN
4,1,2013-01-05,1,4997,NaN,NaN,NaN
5,1,2013-01-06,0,0,NaN,NaN,NaN
6,1,2013-01-07,1,7176,NaN,NaN,NaN
7,1,2013-01-08,1,5580,0.0,NaN,NaN
8,1,2013-01-09,1,5471,5530.0,NaN,NaN
9,1,2013-01-10,1,4892,4327.0,NaN,NaN


In [27]:
df['check_gap'] = df.groupby('Store')['Date'].diff(7).dt.days
df['check_gap'].value_counts()

check_gap
7.0      1008144
191.0       1260
Name: count, dtype: int64

In [28]:
import numpy as np

for lag in [7, 14, 28]:
    gap = df.groupby('Store')['Date'].diff(lag).dt.days
    df.loc[gap != lag, f'Sales_lag_{lag}'] = np.nan

# Kontrol
for lag in [7, 14, 28]:
    print(f'Sales_lag_{lag} eksik sayısı:', df[f'Sales_lag_{lag}'].isnull().sum())

Sales_lag_7 eksik sayısı: 9065
Sales_lag_14 eksik sayısı: 18130
Sales_lag_28 eksik sayısı: 36260


In [29]:
for window in [7, 14, 30]:
    df[f'Sales_rolling_mean_{window}'] = (
        df.groupby('Store')['Sales']
          .shift(1)
          .rolling(window)
          .mean()
          .reset_index(drop=True)
    )
    df[f'Sales_rolling_std_{window}'] = (
        df.groupby('Store')['Sales']
          .shift(1)
          .rolling(window)
          .std()
          .reset_index(drop=True)
    )

df[['Store', 'Date', 'Sales', 'Sales_rolling_mean_7', 'Sales_rolling_std_7']].head(15)

,Store,Date,Sales,Sales_rolling_mean_7,Sales_rolling_std_7
0,1,2013-01-01,0,NaN,NaN
1,1,2013-01-02,5530,NaN,NaN
2,1,2013-01-03,4327,NaN,NaN
3,1,2013-01-04,4486,NaN,NaN
4,1,2013-01-05,4997,NaN,NaN
5,1,2013-01-06,0,NaN,NaN
6,1,2013-01-07,7176,NaN,NaN
7,1,2013-01-08,5580,3788.000000,2752.283961
8,1,2013-01-09,5471,4585.142857,2231.018633
9,1,2013-01-10,4892,4576.714286,2226.961885


In [30]:
for window in [7, 14, 30]:
    gap = df.groupby('Store')['Date'].diff(window).dt.days
    df.loc[gap != window, f'Sales_rolling_mean_{window}'] = np.nan
    df.loc[gap != window, f'Sales_rolling_std_{window}'] = np.nan

for window in [7, 14, 30]:
    print(f'Sales_rolling_mean_{window} eksik sayısı:', df[f'Sales_rolling_mean_{window}'].isnull().sum())

Sales_rolling_mean_7 eksik sayısı: 9065
Sales_rolling_mean_14 eksik sayısı: 18130
Sales_rolling_mean_30 eksik sayısı: 38850


In [31]:
# Hafta sonu flag'i (DayOfWeek 6=Cumartesi, 7=Pazar)
df['IsWeekend'] = df['DayOfWeek'].isin([6, 7]).astype(int)

# Yılbaşına kaç gün kaldığı (EDA'da bulduğumuz Kasım-Aralık artışını doğrudan yakalar)
df['Christmas_this_year'] = pd.to_datetime(df['Year'].astype(str) + '-12-25')
df['DaysToChristmas'] = (df['Christmas_this_year'] - df['Date']).dt.days

mask = df['DaysToChristmas'] < 0
df.loc[mask, 'Christmas_this_year'] = pd.to_datetime((df.loc[mask, 'Year'] + 1).astype(str) + '-12-25')
df.loc[mask, 'DaysToChristmas'] = (df.loc[mask, 'Christmas_this_year'] - df.loc[mask, 'Date']).dt.days

df = df.drop(columns='Christmas_this_year')

# Rakip mağaza bu tarihte gerçekten açık mı
df['CompetitionOpen'] = (
    (df['Year'] > df['CompetitionOpenSinceYear']) |
    ((df['Year'] == df['CompetitionOpenSinceYear']) & (df['Month'] >= df['CompetitionOpenSinceMonth']))
).astype(int)
df.loc[df['CompetitionOpenSinceYear'].isnull(), 'CompetitionOpen'] = 0

df[['Date', 'DayOfWeek', 'IsWeekend', 'DaysToChristmas', 'CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth', 'CompetitionOpen']].sample(10)

,Date,DayOfWeek,IsWeekend,DaysToChristmas,CompetitionOpenSinceYear,CompetitionOpenSinceMonth,CompetitionOpen
621671,2013-07-23,2,0,155,2014.0,7.0,0
155045,2015-03-05,4,0,295,NaN,NaN,0
468341,2013-07-03,3,0,175,2012.0,7.0,1
667643,2015-07-05,7,1,173,1999.0,10.0,1
708866,2013-10-06,7,1,80,2004.0,4.0,1
442387,2013-09-17,2,0,99,NaN,NaN,0
307599,2014-09-26,5,0,90,NaN,NaN,0
46812,2013-08-31,6,1,116,2014.0,4.0,0
800841,2013-03-19,2,0,281,NaN,NaN,0
252080,2013-06-12,3,0,196,2010.0,2.0,1


In [32]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['StoreType', 'Assortment', 'StateHoliday']

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

df[['StoreType', 'StoreType_enc', 'Assortment', 'Assortment_enc', 'StateHoliday', 'StateHoliday_enc']].drop_duplicates().sort_values('StoreType')

,StoreType,StoreType_enc,Assortment,Assortment_enc,StateHoliday,StateHoliday_enc
942,a,0,a,0,a,1
943,a,0,a,0,0,0
1029,a,0,a,0,b,2
1300,a,0,a,0,c,3
1325,a,0,a,0,0,0
5652,a,0,c,2,a,1
5653,a,0,c,2,0,0
5739,a,0,c,2,b,2
6010,a,0,c,2,c,3
6035,a,0,c,2,0,0


In [33]:
print(df['Date'].min(), '-', df['Date'].max())

2013-01-01 00:00:00 - 2015-07-31 00:00:00


In [34]:
split_date = '2015-06-19'

train_df = df[df['Date'] < split_date].copy()
test_df = df[df['Date'] >= split_date].copy()

print('Train:', train_df.shape, train_df['Date'].min(), '-', train_df['Date'].max())
print('Test:', test_df.shape, test_df['Date'].min(), '-', test_df['Date'].max())

Train: (969264, 39) 2013-01-01 00:00:00 - 2015-06-18 00:00:00
Test: (47945, 39) 2015-06-19 00:00:00 - 2015-07-31 00:00:00


In [35]:
train_df = train_df[train_df['Open'] == 1]

feature_check_cols = ['Sales_lag_7', 'Sales_lag_14', 'Sales_lag_28',
                       'Sales_rolling_mean_7', 'Sales_rolling_mean_14', 'Sales_rolling_mean_30']
train_df = train_df.dropna(subset=feature_check_cols)

print('Temizlenmiş Train:', train_df.shape)

Temizlenmiş Train: (770908, 39)


In [ ]:
train_df.to_csv('../data/processed/train_featured.csv', index=False)
test_df.to_csv('../data/processed/test_featured.csv', index=False)
print('Kaydedildi -> Train:', train_df.shape, ' Test:', test_df.shape)
